# 27 - Explanation

## Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path
import joblib

## Paths

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.features.job_recommender import job_recommender

## Get recommendations

In [4]:
skill_text= "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, statistical modelling," \
"r, ecology, visualisaion, ggpplot, seaborn, numpy, pandas, git, github, microsoft office, phd, neural networks, excell, teamwork, team member, cloud, aws" \
"pca, recommender systems, shiny app, shiny, technical writting, scientific research"
current_state= ("ALL")
job_title_family = "data_scientist"
job_title_rich= None
target_sectors = None
salary_target = 200000
explain_skills = False


rec = job_recommender(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills,
                             verbose=True)

Predicting salary based on input skills...
Acceptance shape test passed: True
Acceptance alignment test passed: True
Applying the suitability index threshold 0.7...
* Returning jobs after applying s_min=0.7.
* Suitable jobs identified = 287.
* 1015 filtered out due to low suitability.
Applying competitiveness filter into 2 buckets: "best_now" and "stretch".
Number of "Best-now" jobs = 274
Number of "Stretch" jobs = 13
Computing ranking score based on suitability and competitiveness.
Top 10 "best-now" jobs:

                          Size                      Sector  \
job_id                                                       
821     1001 to 5000 employees      Information Technology   
289          1 to 50 employees      Information Technology   
112     1001 to 5000 employees           Business Services   
797           10000+ employees                   Insurance   
744     1001 to 5000 employees      Information Technology   
895                    Unknown                     Un

In [5]:
top_best = rec["tables"]["top_best_now"].copy()

top_stretch = rec["tables"]["top_stretch"].copy()

params = rec["params"]

skill_gap = rec['tables']['skill_gap']
skill_mat = rec['tables']['skill_prob_matrix']

profile = rec['profile']

df = rec["tables"]["candidate_jobs"]

In [6]:
df

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index,pred_sal,bucket,competitiveness_bucket,score
814,821,Numerator is looking for Sr. Data Scientists t...,3.9,1001 to 5000 employees,2004.0,IT Services,Information Technology,data_scientist,IL,private,...,0.3000,0.720106,0.281648,0.010431,0.057604,0.034018,96481.648438,best_now,best_now,0.703097
286,289,Senior Data Scientist Description The Senior D...,3.6,1 to 50 employees,NaN,IT Services,Information Technology,data_scientist,NJ,private,...,0.5575,0.810655,0.257262,0.009528,0.428187,0.218858,114370.945312,best_now,best_now,0.701226
110,112,As a data scientist at TBWA\Chiat\Day New York...,2.7,1001 to 5000 employees,1968.0,Advertising & Marketing,Business Services,data_scientist,NY,public,...,0.5575,0.807202,0.156264,0.005788,0.428187,0.216987,127717.648438,best_now,best_now,0.698709
2973,2992,Facebook's mission is to give people the power...,4.5,10000+ employees,2004.0,Internet,Information Technology,data_scientist,CA,public,...,1.0000,0.948958,0.315807,0.011697,0.994240,0.502968,147836.468750,stretch,stretch,0.697474
791,797,Aon is looking for a Senior Data Scientist\n\n...,3.5,10000+ employees,1892.0,Insurance Agencies & Brokerages,Insurance,data_scientist,IL,public,...,0.2175,0.704026,0.315547,0.011687,0.016513,0.014100,95511.812500,best_now,best_now,0.696976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,134,"Get To Know Voice\n\nAt Voice, we are on a mis...",3.4,Unknown,NaN,Unknown,Unknown,data_scientist,NY,private,...,0.8475,0.702351,0.066005,0.002445,0.899386,0.450915,126374.179688,best_now,best_now,0.476893
2966,2985,"Introduction\nAs a Data Scientist at IBM, you ...",3.7,10000+ employees,1911.0,IT Services,Information Technology,data_scientist,CA,public,...,1.0000,0.726712,0.309801,0.011474,0.994240,0.502857,126266.281250,stretch,stretch,0.475283
168,171,"Entera, where residential real estate investin...",NaN,Unknown,NaN,Real Estate,Real Estate,data_scientist,NY,private,...,1.0000,0.720423,0.493329,0.018271,0.980031,0.499151,129609.265625,best_now,best_now,0.470847
2537,2553,Position Role/Tile: Data Scientist\nLocation: ...,3.0,51 to 200 employees,NaN,Consulting,Business Services,data_scientist,TX,private,...,0.8600,0.700770,0.063894,0.002366,0.922811,0.462589,100532.257812,best_now,best_now,0.469476


## Add deterministic explanation fields

In [7]:
# Why bucket
c_max = float(params["c_max"])

top_best["why_bucket"] = top_best["competitiveness_index"].map(
    lambda x: f"Barrier is low (competitiveness {x:.2f} ≤ {c_max:.2f})"
)

top_stretch["why_bucket"] = top_stretch["competitiveness_index"].map(
    lambda x: f"Barrier is higher (competitiveness {x:.2f} > {c_max:.2f})"
)

# why rank
alpha = float(params["alpha"])
top_best["why_rank"] = (
    "Ranked by score = "
    + top_best["suitability"].map(lambda v: f"{v:.2f}")
    + " - ("
    + f"{alpha:.2f}"
    + " * "
    + top_best["competitiveness_index"].map(lambda v: f"{v:.2f}")
    + ") = "
    + top_best["score"].map(lambda v: f"{v:.2f}")
)
top_stretch['why_rank'] = (
    "Ranked by score = "
    + top_stretch["suitability"].map(lambda v: f"{v:.2f}")
    + " - ("
    + f"{alpha:.2f}"
    + " * "
    + top_stretch["competitiveness_index"].map(lambda v: f"{v:.2f}")
    + ") = "
    + top_stretch["score"].map(lambda v: f"{v:.2f}")
)

# Salary context
sal_mean_best = pd.to_numeric(top_best["sal_mean"], errors="coerce")
pred_sal_best = pd.to_numeric(top_best["pred_sal"], errors="coerce")
gap_best = sal_mean_best - pred_sal_best


sal_mean_stretch = pd.to_numeric(top_stretch["sal_mean"], errors="coerce")
pred_sal_stretch = pd.to_numeric(top_stretch["pred_sal"], errors="coerce")
gap_stretch = sal_mean_stretch - pred_sal_stretch

assert sal_mean_best.notna().all() and pred_sal_best.notna().all()
assert sal_mean_stretch.notna().all() and pred_sal_stretch.notna().all()

top_best["salary_context"] = (
    "Market mean = " + sal_mean_best.map(lambda v: f"{v:.2f}")
    + "; predicted for you = " + pred_sal_best.map(lambda v: f"{v:.2f}")
    + "; gap (market - predicted) = " + gap_best.map(lambda v: f"{v:.2f}")
)

top_stretch["salary_context"] = (
    "Market mean = " + sal_mean_stretch.map(lambda v: f"{v:.2f}")
    + "; predicted for you = " + pred_sal_stretch.map(lambda v: f"{v:.2f}")
    + "; gap (market - predicted) = " + gap_stretch.map(lambda v: f"{v:.2f}")
)


## Skill strengths

In [8]:
# Add skill metrics
gap_addon = rec["tables"]["candidate_jobs"][["job_id", "skill_match_norm", "expected_missing_norm"]]
top_best = top_best.merge(gap_addon, how = 'left', on = 'job_id')
top_stretch = top_stretch.merge(gap_addon, how = 'left', on = 'job_id')

In [9]:
user_skill_vector = profile['derived']['skill_vector']

In [10]:
tau = 0.5
prob_cols = [c for c in skill_mat.columns if c.endswith("_prob")]
req_matrix = skill_mat[["job_id"] + prob_cols].copy()
req_matrix[prob_cols] = (req_matrix[prob_cols] >= tau).astype(int)
rename_map = {f"{fam}_prob": fam for fam in user_skill_vector.columns}
req_matrix = req_matrix.rename(columns=rename_map)
missing = set(user_skill_vector.columns) - set(req_matrix.columns)

In [11]:
missing

set()

In [12]:
u = user_skill_vector.iloc[0]
R = req_matrix.set_index("job_id")[u.index].astype(bool)
missing_mask = R & (~u.astype(bool))
covered_mask = R & (u.astype(bool))

In [13]:
missing_families = missing_mask.apply(lambda r: r.index[r].tolist(), axis=1).rename('missing_families').reset_index()
covered_families = covered_mask.apply(lambda r: r.index[r].tolist(), axis=1).rename('covered_families').reset_index()

In [14]:
top_best = top_best.merge(missing_families, how = 'left', on = 'job_id')
top_best = top_best.merge(covered_families, how = 'left', on = 'job_id')
top_stretch = top_stretch.merge(missing_families, how = 'left', on = 'job_id')
top_stretch = top_stretch.merge(covered_families, how = 'left', on = 'job_id')

In [15]:
top_best["missing_families"].map(len).value_counts().head()

missing_families
2    5
3    3
1    1
4    1
Name: count, dtype: int64

In [16]:
top_stretch["missing_families"].map(len).value_counts().head()

missing_families
4    2
2    1
7    1
5    1
Name: count, dtype: int64

In [17]:
top_best

,job_id,Size,Sector,Industry,state,title_rich,sal_mean,pred_sal,suitability,competitiveness_index,score,bucket,why_bucket,why_rank,salary_context,skill_match_norm,expected_missing_norm,missing_families,covered_families
0,821,1001 to 5000 employees,Information Technology,IT Services,IL,general_data_data_scientist,60000.0,96481.648438,0.720106,0.034018,0.703097,best_now,Barrier is low (competitiveness 0.03 ≤ 0.50),Ranked by score = 0.72 - (0.50 * 0.03) = 0.70,Market mean = 60000.00; predicted for you = 96...,0.900152,0.010431,"[db_storage__intermediate, soft_skills__leader...","[core_programming__basic, ml_ai__basic, ml_ai_..."
1,289,1 to 50 employees,Information Technology,IT Services,NJ,general_data_data_scientist,111500.0,114370.945312,0.810655,0.218858,0.701226,best_now,Barrier is low (competitiveness 0.22 ≤ 0.50),Ranked by score = 0.81 - (0.50 * 0.22) = 0.70,Market mean = 111500.00; predicted for you = 1...,0.919150,0.009528,"[data_engineering_pipelines__intermediate, dat...","[core_programming__basic, ml_ai__basic, ml_ai_..."
2,112,1001 to 5000 employees,Business Services,Advertising & Marketing,NY,general_data_data_scientist,111500.0,127717.648438,0.807202,0.216987,0.698709,best_now,Barrier is low (competitiveness 0.22 ≤ 0.50),Ranked by score = 0.81 - (0.50 * 0.22) = 0.70,Market mean = 111500.00; predicted for you = 1...,0.914218,0.005788,"[data_engineering_pipelines__intermediate, dat...","[core_programming__basic, ml_ai__basic, ml_ai_..."
3,797,10000+ employees,Insurance,Insurance Agencies & Brokerages,IL,general_data_data_scientist,43500.0,95511.812500,0.704026,0.014100,0.696976,best_now,Barrier is low (competitiveness 0.01 ≤ 0.50),Ranked by score = 0.70 - (0.50 * 0.01) = 0.70,Market mean = 43500.00; predicted for you = 95...,0.912537,0.011687,"[analytics_stats__intermediate, db_storage__in...","[core_programming__basic, ml_ai__basic, ml_ai_..."
4,744,1001 to 5000 employees,Information Technology,IT Services,IL,general_data_data_scientist,104500.0,96481.648438,0.794453,0.195790,0.696558,best_now,Barrier is low (competitiveness 0.20 ≤ 0.50),Ranked by score = 0.79 - (0.50 * 0.20) = 0.70,Market mean = 104500.00; predicted for you = 9...,0.911004,0.013316,"[db_storage__intermediate, domain_specific__none]","[core_programming__basic, ml_ai__basic, ml_ai_..."
5,895,Unknown,Unknown,Unknown,IL,security_data_scientist,111500.0,88731.484375,0.804953,0.222712,0.693597,best_now,Barrier is low (competitiveness 0.22 ≤ 0.50),Ranked by score = 0.80 - (0.50 * 0.22) = 0.69,Market mean = 111500.00; predicted for you = 8...,0.911004,0.017237,"[data_engineering_pipelines__intermediate, db_...","[core_programming__basic, ml_ai__basic, ml_ai_..."
6,1967,501 to 1000 employees,Business Services,Consulting,TX,general_data_data_scientist,125000.0,99382.179688,0.829627,0.275656,0.691798,best_now,Barrier is low (competitiveness 0.28 ≤ 0.50),Ranked by score = 0.83 - (0.50 * 0.28) = 0.69,Market mean = 125000.00; predicted for you = 9...,0.917324,0.009838,"[db_storage__intermediate, soft_skills__leader...","[core_programming__basic, ml_ai__basic, ml_ai_..."
7,267,1 to 50 employees,Unknown,Unknown,NJ,general_data_data_scientist,124500.0,110793.429688,0.823830,0.269052,0.689304,best_now,Barrier is low (competitiveness 0.27 ≤ 0.50),Ranked by score = 0.82 - (0.50 * 0.27) = 0.69,Market mean = 124500.00; predicted for you = 1...,0.910114,0.008533,"[data_engineering_pipelines__advanced, soft_sk...","[core_programming__basic, ml_ai__basic, ml_ai_..."
8,3387,501 to 1000 employees,Education,Education Training Services,TX,general_data_data_scientist,92000.0,98460.070312,0.753517,0.135063,0.685985,best_now,Barrier is low (competitiveness 0.14 ≤ 0.50),Ranked by score = 0.75 - (0.50 * 0.14) = 0.69,Market mean = 92000.00; predicted for you = 98...,0.879310,0.004765,[data_engineering_pipelines__intermediate],"[core_programming__basic, ml_ai__basic, ml_ai_..."
9,4994,1001 to 5000 employees,Transportation & Logistics,Express Delivery Services,PA,general_data_data_scie

## Test explanator

In [18]:
from src.job_intel.features.job_explanations import build_job_explanations

In [19]:
skill_text= "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, statistical modelling," \
"r, ecology, visualisaion, ggpplot, seaborn, numpy, pandas, git, github, microsoft office, phd, neural networks, excell, teamwork, team member, cloud, aws" \
"pca, recommender systems, shiny app, shiny, technical writting, scientific research"
current_state= ("ALL")
job_title_family = "data_scientist"
job_title_rich= None
target_sectors = None
salary_target = 200000
explain_skills = False


rec = job_recommender(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills,
                             verbose=True)

Predicting salary based on input skills...
Acceptance shape test passed: True
Acceptance alignment test passed: True
Applying the suitability index threshold 0.7...
* Returning jobs after applying s_min=0.7.
* Suitable jobs identified = 287.
* 1015 filtered out due to low suitability.
Applying competitiveness filter into 2 buckets: "best_now" and "stretch".
Number of "Best-now" jobs = 274
Number of "Stretch" jobs = 13
Computing ranking score based on suitability and competitiveness.
Top 10 "best-now" jobs:

                          Size                      Sector  \
job_id                                                       
821     1001 to 5000 employees      Information Technology   
289          1 to 50 employees      Information Technology   
112     1001 to 5000 employees           Business Services   
797           10000+ employees                   Insurance   
744     1001 to 5000 employees      Information Technology   
895                    Unknown                     Un

In [20]:
top_best = rec["tables"]["top_best_now"].copy()

top_stretch = rec["tables"]["top_stretch"].copy()

params = rec["params"]

skill_gap = rec['tables']['skill_gap']
skill_mat = rec['tables']['skill_prob_matrix']

profile = rec['profile']
df = rec["tables"]["candidate_jobs"]

In [21]:
res = build_job_explanations(rec = rec)

In [22]:
res['tables']

{'top_best_explained':    job_id                    Size                      Sector  \
 0     821  1001 to 5000 employees      Information Technology   
 1     289       1 to 50 employees      Information Technology   
 2     112  1001 to 5000 employees           Business Services   
 3     797        10000+ employees                   Insurance   
 4     744  1001 to 5000 employees      Information Technology   
 5     895                 Unknown                     Unknown   
 6    1967   501 to 1000 employees           Business Services   
 7     267       1 to 50 employees                     Unknown   
 8    3387   501 to 1000 employees                   Education   
 9    4994  1001 to 5000 employees  Transportation & Logistics   
 
                           Industry state                   title_rich  \
 0                      IT Services    IL  general_data_data_scientist   
 1                      IT Services    NJ  general_data_data_scientist   
 2          Advertising & Ma

# == End of Notebook ==